# Commented notebook: using `print_peetre_decomposition`

This notebook rewrites the original examples with Markdown explanations and Python comments.
It focuses on the method `PseudoDifferentialOperator.print_peetre_decomposition` from `psiop.py`.

The method prints a symbolic Peetre-style decomposition of the operator symbol into:

1. local polynomial terms in the frequency variables,
2. separable non-local terms of the form `a(x) * q(xi)`,
3. a joint residual part that is genuinely entangled in space and frequency.


## What `print_peetre_decomposition` does

The method is mainly a pretty-printing utility. Internally it calls `peetre_decomposition`
and then displays the resulting pieces in a readable format.

Its most important options are:

- `joint_backend='direct'`: print the raw joint residual terms.
- `joint_backend='lowrank'`: approximate the joint residual by a short sum of separable pairs
  `a_k(x) * q_k(xi)` using Chebyshev/SVD factorization.
- `joint_bounds`: required when `joint_backend='lowrank'`, because the low-rank approximation
  is only defined on a bounded phase-space rectangle.
- `joint_degree`: Chebyshev degree used in each variable for the low-rank approximation.
- `joint_tol`: tolerance for singular-value truncation and coefficient pruning.
- `joint_num_samples`: number of Monte Carlo samples used to estimate approximation quality.
- `joint_seed`: random seed for reproducible Monte Carlo diagnostics.
- `separable_local`: passed through `**kwargs` to `peetre_decomposition`.


## Imports

We import SymPy, the `PseudoDifferentialOperator` class, and `time` for timing the 2D examples.


In [1]:
import time

import sympy as sp
from psiop import PseudoDifferentialOperator


## 1D example: local, separable, and joint terms

We build a one-dimensional symbol containing three different types of terms:

- `xi**2` is local because it is polynomial in the frequency variable.
- `x*sp.sin(xi)` is separable because it is a product of a spatial factor and a frequency factor.
- `1 / (1 + (x - xi)**2)` is genuinely joint because `x` and `xi` are entangled.


In [2]:
# Define the 1D spatial variable x and the frequency variable xi.
x, xi = sp.symbols('x xi', real=True)

# Build a 1D symbol with three different Peetre classes:
#   1. xi**2: local polynomial part.
#   2. x*sin(xi): separable non-local part, a(x) * q(xi).
#   3. 1/(1 + (x - xi)**2): genuinely joint term.
p1 = xi**2 + x*sp.sin(xi) + 1 / (1 + (x - xi)**2)

# mode='symbol' tells psiop that p1 is already the phase-space symbol p(x, xi).
op1 = PseudoDifferentialOperator(p1, [x], mode='symbol')


## Default printing: direct joint residual

With no arguments, `print_peetre_decomposition` uses:

- `joint_backend='direct'`
- `separable_local=False`

Therefore the joint residual is printed as a raw symbolic expression, without low-rank factorization.


In [3]:
print('=' * 70)
print('DEFAULT: joint_backend=direct, separable_local=False')
print('=' * 70)

# Default call.
# This prints:
#   - local terms,
#   - separable non-local terms,
#   - irreducible joint terms,
#   - the reconstructed local, separable, and joint symbols.
op1.print_peetre_decomposition()


DEFAULT: joint_backend=direct, separable_local=False
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## Option: `separable_local=True`

The keyword `separable_local` is not a direct named parameter of `print_peetre_decomposition`,
but it is forwarded to `peetre_decomposition` through `**kwargs`.

When `separable_local=True`, local polynomial terms are exposed in the same operational form
as separable terms, namely as pairs `a(x) * q(xi)`.


In [4]:
print('separable_local=True')

# Here separable_local is passed through **kwargs to peetre_decomposition.
# Local polynomial terms are then displayed in separable-pair form.
op1.print_peetre_decomposition(separable_local=True)


separable_local=True
--- 0 local term(s), polynomial in (xi,) ---
--- 2 separable non-local term(s) ---
  (1) * (xi**2)
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = 0
separable_symbol = x*sin(xi) + xi**2
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## Option: `joint_backend='lowrank'`

When `joint_backend='lowrank'`, the joint residual is not printed raw.
Instead, it is approximated on a bounded rectangle by a short sum of separable terms:

```
p_joint(x, xi) ~= sum_k a_k(x) q_k(xi)
```

Because this approximation is only meaningful on a bounded domain, `joint_bounds` is required.


In [5]:
print('joint_backend=lowrank')

# Low-rank printing requires explicit bounds for every space and frequency variable.
# Here the approximation is performed on:
#   x in [-5, 5]
#   xi in [-30, 30]
op1.print_peetre_decomposition(
    joint_backend='lowrank',
    joint_bounds={x: (-5, 5), xi: (-30, 30)},
    joint_degree=6,
)


joint_backend=lowrank
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual factorized into 5 low-rank term(s) via factorize_symbolic (rel_l2_error=1.432e+00) ---
  (0.00026926*x**6 - 0.013732*x**4 + 0.21358*x**2 - 1.0019) * (7.2629e-9*xi**6 - 1.3078e-5*xi**4 + 0.0069955*xi**2 - 0.99805)
  (1.0742e-5*x**6 - 0.00054136*x**4 + 0.0091446*x**2 + 0.011449) * (9.7625e-10*xi**6 - 1.5949e-6*xi**4 + 0.00065437*xi**2 + 0.0093696)
  (6.8068e-7*x**5 + 7.4843e-5*x**3 + 0.0091005*x) * (1.1576e-8*xi**5 - 1.9081e-5*xi**3 + 0.0080346*xi)
  (5.5743e-6*x**6 - 0.00029603*x**4 + 0.0036664*x**2 - 0.0026623) * (-1.6396e-10*xi**6 + 2.3675e-7*xi**4 - 7.4572e-5*xi**2 + 0.0047266)
  (-1.3656e-6*x**5 - 0.00010283*x**3 + 0.0025124*x) * (-1.3111e-9*xi**5 + 1.7515e-6*xi**3 - 0.00039665*xi)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


## Smoother joint kernel in 1D

The previous joint term was a rational function. Here we use a Gaussian joint kernel,
which is smoother and often easier to approximate by a low-rank separable expansion.


In [6]:
# Redefine symbols to keep the example self-contained.
x, xi = sp.symbols('x xi', real=True)

# A Gaussian bump exp(-(x - xi)^2 / 8) is a smooth joint kernel.
p1b = xi**2 + x*sp.sin(xi) + sp.exp(-((x - xi)**2) / 8)

# Create the operator from the explicit symbol.
op1b = PseudoDifferentialOperator(p1b, [x], mode='symbol')


## Comparing bounds and degrees for the low-rank approximation

The printed diagnostic `rel_l2_error` is a Monte Carlo estimate of the approximation error
of the low-rank separable representation of the joint residual.

Increasing `joint_degree` usually improves the approximation, but also increases the cost.


In [7]:
# Try two bounded windows and two Chebyshev degrees.
for bounds, deg in [
    ({x: (-5, 5), xi: (-15, 15)}, 8),
    ({x: (-4, 4), xi: (-12, 12)}, 10),
]:
    print('bounds =', bounds, ', degree =', deg)

    # Print the low-rank factorization of the joint residual.
    op1b.print_peetre_decomposition(
        joint_backend='lowrank',
        joint_bounds=bounds,
        joint_degree=deg,
    )

    print()


bounds = {x: (-5, 5), xi: (-15, 15)} , degree = 8
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual factorized into 5 low-rank term(s) via factorize_symbolic (rel_l2_error=4.352e-01) ---
  (-3.4432e-7*x**7 + 0.00017519*x**5 - 0.0078706*x**3 - 0.026926*x) * (5.4597e-8*xi**7 - 2.9021e-5*xi**5 + 0.0049665*xi**3 - 0.27015*xi)
  (-2.0444e-6*x**8 + 0.00017141*x**6 - 0.0059783*x**4 + 0.11081*x**2 - 0.8798) * (-8.2942e-9*xi**8 + 4.6243e-6*xi**6 - 0.00087012*xi**4 + 0.0611*xi**2 - 1.1139)
  (9.7261e-7*x**8 - 8.6251e-5*x**6 + 0.0021714*x**4 + 0.0032102*x**2 + 0.15204) * (-8.3651e-9*xi**8 + 4.4135e-6*xi**6 - 0.00074419*xi**4 + 0.038716*xi**2 + 0.12941)
  (2.266e-6*x**7 + 1.4903e-5*x**5 - 0.0011314*x**3 - 0.0048597*x) * (8.219e-9*xi**7 - 3.7058e-6*xi**5 + 0.00047073*xi**3 - 0.011925*xi)
  (-1.6054e-7*x**8 - 9.6843e-6*x**6 + 6.9195e-5*x**4 + 0.0054866*x**2 - 0.015388) * (-9.3695e-10*xi**8 + 4.3579e-7*xi**6 -

## 2D example

The same API works in two spatial dimensions.

The symbol below contains:

- a local polynomial part: `xi**2 + eta**2`,
- a separable part: `x*y*cos(xi + eta)`,
- a joint Gaussian residual entangling `(x, y)` with `(xi, eta)`.


In [8]:
# Define 2D spatial variables and frequency variables.
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# Build a 2D symbol with local, separable, and joint pieces.
p2 = (
    xi**2
    + eta**2
    + x*y*sp.cos(xi + eta)
    + sp.exp(-((x - xi)**2 + (y - eta)**2) / 8)
)

# Create the 2D pseudo-differential operator from its symbol.
op2 = PseudoDifferentialOperator(p2, [x, y], mode='symbol')


## Default 2D printing

As in 1D, the default backend prints the raw joint residual.


In [9]:
print('DEFAULT 2D: joint_backend=direct')

# In 2D, local terms are polynomial in the pair (xi, eta).
op2.print_peetre_decomposition()


DEFAULT 2D: joint_backend=direct
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- 1 irreducible joint term(s) ---
  exp(-eta**2/8)*exp(-x**2/8)*exp(-xi**2/8)*exp(-y**2/8)*exp(eta*y/4)*exp(x*xi/4)
local_symbol = eta**2 + xi**2
separable_symbol = x*y*cos(eta + xi)
joint_symbol = exp(-eta**2/8)*exp(-x**2/8)*exp(-xi**2/8)*exp(-y**2/8)*exp(eta*y/4)*exp(x*xi/4)


## 2D low-rank joint printing and timing

In 2D, the Chebyshev grid size grows quickly because the tensor product has four variables:
`x`, `y`, `xi`, and `eta`.

For that reason, we use small degrees and a smaller Monte Carlo sample size.


In [11]:
# Try small Chebyshev degrees and measure the wall-clock time.
for deg in [2, 3, 4]:
    t0 = time.time()

    # Low-rank printing in 2D requires bounds for x, y, xi, and eta.
    op2.print_peetre_decomposition(
        joint_backend='lowrank',
        joint_bounds={x: (-3, 3), y: (-3, 3), xi: (-3, 3), eta: (-3, 3)},
        joint_degree=deg,
        joint_num_samples=3000,
    )

    print('degree', deg, 'time', time.time() - t0)
    print()


--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- joint residual factorized into 9 low-rank term(s) via factorize_symbolic (rel_l2_error=2.859e-01) ---
  (-0.0029295*x**2*y**2 + 0.052254*x**2 + 0.052254*y**2 - 0.93207) * (-0.0029295*eta**2*xi**2 + 0.052254*eta**2 + 0.052254*xi**2 - 0.93207)
  (-0.011264*x**2*y + 0.0058369*x*y**2 - 0.10411*x + 0.20091*y) * (0.0058369*eta**2*xi - 0.011264*eta*xi**2 + 0.20091*eta - 0.10411*xi)
  (0.0058369*x**2*y + 0.011264*x*y**2 - 0.20091*x - 0.10411*y) * (0.011264*eta**2*xi + 0.0058369*eta*xi**2 - 0.10411*eta - 0.20091*xi)
  (-0.054938*x*y) * (-0.054938*eta*xi)
  (-4.9427e-5*x**2*y**2 - 0.049184*x**2 + 0.050213*y**2 - 0.0026288) * (-4.9427e-5*eta**2*xi**2 + 0.050213*eta**2 - 0.049184*xi**2 - 0.0026288)
  (0.006691*x**2*y**2 - 0.070016*x**2 - 0.069282*y**2 + 0.35586) * (0.006691*eta**2*xi**2 - 0.069282*eta**2 - 0.070016*xi**2 + 0.35586)
  (0.0085603*x**2*y

## Parameter summary

| Parameter | Typical value | Meaning |
|---|---:|---|
| `joint_backend` | `'direct'` | Print the raw joint residual. |
| `joint_backend` | `'lowrank'` | Factor the joint residual into separable pairs. |
| `joint_bounds` | `{x: (-5, 5), xi: (-30, 30)}` | Bounded domain for low-rank factorization. |
| `joint_degree` | `6` | Chebyshev degree per variable. |
| `joint_tol` | `1e-5` | Singular-value and coefficient pruning tolerance. |
| `joint_num_samples` | `10000` | Monte Carlo samples for error diagnostics. |
| `joint_seed` | `42` | Random seed for reproducible diagnostics. |
| `separable_local` | `False` or `True` | Passed to `peetre_decomposition`; controls local-term representation. |
| `use_cache` | `True` | Cache the underlying Peetre decomposition when possible. |
